# thui-avo-v1 (Thuitanium / Knowless Crew) — the AVO bundle plus two breakers on its loop

`thui-avo-v0` (Tufa's AVO agent as shipped) plus ONE cell-12 wrap on `AvoAgent.analyze`, written
from the 09-05 full run's own transcripts: **(A)** a game that makes K analyze() calls in a row
without executing an action gets ONE simple action executed through the session's `step_env`
(m0r0 made 189 tool calls in 93 turns and never called `action()`; the supervisor's 30 paragraphs
changed nothing); **(B)** two GAME_OVERs inside exploit mode switch exploit mode off for that game
(ls20 replayed one dying plan 28 times, 1,227 actions on level 2). Prompts, memory, supervisor,
chassis and seed are v0's, unchanged.

Solver credit: Tufa Labs (Harold Bessis, Jeroen Cottaar, Isaiah Pressman, Andries Smit,
Michal Tesnar, Stefano Viel) — their AVO agent from their attached dataset. This is a
Knowless Crew / Thuitanium fork; none of their scores are ours.


## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["jakobbrggen/taaf-kaggle-source", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "jakobbrggen/qwen3-8-27b-fp8-hf-snapshot"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.

In [ ]:
import re
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    # duckv10: model swap only (R12 seam). NO output cap - v9 proved a 768-token ceiling
    # truncates the tool call that carries the action itself.
    command = (
        command
        .replace("MODEL_OWNER = 'driessmit1'", "MODEL_OWNER = 'jakobbrggen'")
        .replace(
            "MODEL_SLUG = 'vrfai-qwen3-6-27b-fp8-hf-snapshot'",
            "MODEL_SLUG = 'qwen3-8-27b-fp8-hf-snapshot'",
        )
        .replace(
            "SERVED_MODEL_NAME = 'vrfai/Qwen3.6-27B-FP8'",
            "SERVED_MODEL_NAME = 'vrfai/Qwen3.8-27B-FP8'",
        )
        .replace(
            "    'LOCAL_ANALYZER_TEMPERATURE':",
            "    'LOCAL_ANALYZER_SEED': '20260825',\n    'LOCAL_ANALYZER_TEMPERATURE':",
        )
    )
    assert "vrfai-qwen3-6-27b-fp8-hf-snapshot" not in command, "duckv10: model slug rewrite missed"
    assert "Qwen3.6-27B-FP8" not in command, "duckv10: served-name rewrite missed"
    assert "'LOCAL_ANALYZER_MAX_OUTPUT': '0'" in command, "duckv10: output must stay UNCAPPED"
    # thui-v1-1 TEETH, in-kernel, before the benchmark starts. The seed IS this
    # build: if the anchor moved in a newer bundle the replace is a silent no-op and the
    # run would score normally while measuring nothing.
    assert "'LOCAL_ANALYZER_SEED': '20260825'" in command, (
        "thui-v1-1 TEETH FAIL: seed injection missed -- the setup_env anchor "
        "\"    'LOCAL_ANALYZER_TEMPERATURE':\" is not in this bundle's setup command"
    )
    assert command.count("'LOCAL_ANALYZER_SEED'") == 1, (
        "thui-v1-1 TEETH FAIL: seed key injected more than once"
    )
    assert "'LOCAL_ANALYZER_TEMPERATURE': '0.6'" in command, (
        "thui-v1-1 TEETH FAIL: temperature is not 0.6 -- this arm must not touch it"
    )
    _ups = re.search(r"'MULTIMODAL_UPSCALE': '([^']*)'", command)
    print(f"thui-avo-v0: upstream MULTIMODAL_UPSCALE={_ups.group(1) if _ups else 'ABSENT'} "
          "(as shipped -- deliberately not pinned to 4)", flush=True)
    print("thui-v1-1: sampler pinned, seed=20260825, temperature untouched", flush=True)
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 6. Customization hook

Optional: tweak `bm`, `bm.games`, or `bm.solver` here before the run starts — the safe place
for one-off experiments once the deployed bundle has loaded.

In [ ]:
# === per-request usage probe: completion_tokens + finish_reason + wall time ===
# Runs after `bm`/`bm.solver` are unpickled (cell 10) and before `bm.run(...)` (cell 14),
# so every ToolAgent built during the run is already wrapped -- the analyzer is
# constructed per game inside bm.run (solver.py:1339 _make_analyzer).
#
# NOT `bm.solver.save_request_logs = True`: that flag writes the full message list on
# both the request and the response event (~150-215 MB for this run) and never writes
# `usage`, which is the one field that can place an output-token cap. duckv9 capped at
# 768 without that distribution and scored 0.22 (finish_reason `length` 704 against
# `tool_calls` 68). See docs/audit/2026-08-23-request-usage-probe.md.

from inference.agent import tool_agent

"""Per-request usage probe for the duck harness — the cell-12 payload.

WHY THIS EXISTS, and why it is not `save_request_logs=True`.

The harness already has a per-request logger. `HarnessSolver.save_request_logs`
(`solver.py:885`) flows to `ToolAgent` (`solver.py:1350`) and gates two
`_append_request_snapshot` calls (`tool_agent.py:2195` and `:2208`). Setting
`bm.solver.save_request_logs = True` in cell 14 works — the analyzer is built per game
inside `bm.run()` (`solver.py:1339 _make_analyzer`), so the flag is read after cell 12
and after cell 14's assignments.

It records the wrong thing for the question in front of us. `_append_request_snapshot`
(`tool_agent.py:929-965`) writes `messages` + `tools` + `finish_reason` + `analysis_step`
+ `action` + `request_index_within_turn`. It does **not** write `usage`, even though the
caller holds it — `result.usage` is passed to `_accumulate_usage_tokens` on the line
between the two snapshot calls (`:2207`) and then dropped. And it writes the full message
list on BOTH the request and the response event, so the response row costs ~60-90 KB to
carry one string. At 1,201 requests × 2 events that is roughly 150-215 MB to obtain 1,201
values of `finish_reason`.

The open question is where to place an output-token cap. `duckv9` capped at 768 and scored
0.22 (`finish_reason` `length` 704 against `tool_calls` 68 — the cap truncated the tool call
carrying the action). `duckv10` runs uncapped at a mean 2,211 output tokens per request. So
we know 91% of requests want more than 768 tokens and nothing about the shape above it.
Placing a safe cap needs the **distribution**, which means `completion_tokens` per request.
That is what this file records, at about 200 bytes per request instead of 150 KB.

⚠️ ANSWERED — and the 91% above is the PRIOR this probe was built to test, not a result.
Measured, the share is **68.4%**. The whole paragraph is left standing on purpose: it is the
reason the probe exists, and overwriting an assumption with its answer deletes the record of
what was assumed. Read `notes/R35-usage-distribution.md` for what the run actually found —
§"Two corrections to what was assumed when the probe was written" (`:50-51`) carries this
correction in the author's own words, and the section above it kills the cap idea
structurally: the distribution has **no fat tail**, so every cap that saves meaningfully cuts
the body. A cap at 8,192 saves 0.98% of output; at 12,288 it saves nothing at all.

WHAT IT RECORDS — one JSONL row per request, written next to the run's other artifacts
using the harness's own path convention (`<game>_usage.jsonl`):

    game, action, req_in_turn, wall_s, prompt_tokens, completion_tokens, total_tokens,
    finish_reason

`req_in_turn` is maintained here rather than read from the harness, because the counter the
harness keeps (`turn_count`) is a local in `analyze()`.

WHAT IT ANSWERS

  - the output-token distribution, hence a cap that trims the tail without truncating
    the median (`duckv9`'s failure mode);
  - requests per turn, and how that moves if `LOCAL_ANALYZER_YIELD_SECONDS` changes —
    the tie-breaker for whether raising the yield budget buys anything at all;
  - measured per-request wall time per game, against the 164 s the vLLM log implies.

CONSTRAINTS

  - stdlib only, no project imports beyond `inference.agent.tool_agent`, no top-level side
    effects — `install()` must be called explicitly. The notebook build step embeds this
    file's source into cell 12 and appends the `install(...)` call, the way duckmod splices
    `duck_tools.py`.
  - **fail-open, always.** This is instrument code riding on a run that costs a GPU slot.
    Every hook swallows its own exceptions and the probe disables itself rather than
    letting a logging bug end a game.
  - bounded: `MAX_ROWS_PER_GAME` caps output so a pathological loop cannot fill the disk.
"""

import json
import time

MAX_ROWS_PER_GAME = 2000

_state = {"installed": False, "orig_analyze": None, "orig_chat": None, "disabled": False}


def _usage_path(tool_agent, state_path):
    """Per-game JSONL path, using the harness's own artifact-location convention.

    Falls back to a sibling of the state file if the private resolver is not where we
    expect it — a bundle refresh may move it, and a moved resolver must not end a run.
    """
    try:
        return tool_agent._resolve_named_run_artifact(
            state_path, default_name="request_usage.jsonl", per_game_suffix="_usage.jsonl"
        )
    except Exception:
        return state_path.parent / (state_path.stem + "_usage.jsonl")


def _as_int(value):
    try:
        return max(0, int(value))
    except (TypeError, ValueError):
        return None


def install(tool_agent, *, max_rows=MAX_ROWS_PER_GAME):
    """Wrap ToolAgent.analyze and ToolAgent._chat_completion. Idempotent.

    `tool_agent` is the module (or any object exposing `ToolAgent`), passed in rather than
    imported so this file is testable against a stub without the harness installed.
    """
    if _state["installed"]:
        return False
    agent_cls = tool_agent.ToolAgent
    orig_analyze = agent_cls.analyze
    orig_chat = agent_cls._chat_completion

    def analyze(self, state_path, action_num, *args, **kwargs):
        # Establishes the per-turn context the _chat_completion wrapper stamps onto rows.
        # A turn is one analyze() call; requests within it are numbered from 1.
        try:
            self._probe_path = _usage_path(tool_agent, state_path)
            self._probe_action = action_num
            self._probe_req = 0
        except Exception:
            self._probe_path = None
        return orig_analyze(self, state_path, action_num, *args, **kwargs)

    def _chat_completion(self, messages, **kwargs):
        started = time.monotonic()
        try:
            result = orig_chat(self, messages, **kwargs)
        except Exception as exc:
            _record(self, started, None, type(exc).__name__, max_rows)
            raise
        _record(self, started, result, None, max_rows)
        return result

    agent_cls.analyze = analyze
    agent_cls._chat_completion = _chat_completion
    _state.update(installed=True, orig_analyze=orig_analyze, orig_chat=orig_chat)
    return True


def _record(agent, started, result, exception_name, max_rows):
    """Append one row. Never raises; on any failure the probe turns itself off."""
    if _state["disabled"]:
        return
    try:
        path = getattr(agent, "_probe_path", None)
        if path is None:
            return
        count = getattr(agent, "_probe_rows", 0)
        if count >= max_rows:
            return
        agent._probe_rows = count + 1
        agent._probe_req = getattr(agent, "_probe_req", 0) + 1

        usage = getattr(result, "usage", None) if result is not None else None
        usage = usage if isinstance(usage, dict) else {}
        row = {
            "game": path.stem[: -len("_usage")] if path.stem.endswith("_usage") else path.stem,
            "action": getattr(agent, "_probe_action", None),
            "req_in_turn": agent._probe_req,
            "wall_s": round(time.monotonic() - started, 3),
            "prompt_tokens": _as_int(usage.get("prompt_tokens")),
            "completion_tokens": _as_int(usage.get("completion_tokens")),
            "total_tokens": _as_int(usage.get("total_tokens")),
            "finish_reason": (
                "__exception__:" + exception_name
                if exception_name
                else str(getattr(result, "finish_reason", "") or "")
            ),
        }
        with open(path, "a", encoding="utf-8") as handle:
            handle.write(json.dumps(row, ensure_ascii=True))
            handle.write("\n")
    except Exception:
        # One broken write must not cost a game. Stop trying, keep playing.
        _state["disabled"] = True


def uninstall(tool_agent):
    """Restore the original methods. Exists for the tests, not for the notebook."""
    if not _state["installed"]:
        return False
    tool_agent.ToolAgent.analyze = _state["orig_analyze"]
    tool_agent.ToolAgent._chat_completion = _state["orig_chat"]
    _state.update(installed=False, orig_analyze=None, orig_chat=None, disabled=False)
    return True

_installed = install(tool_agent)
print(f"request-usage probe installed: {_installed}")

# ======================================================================================
# thui-avo-v1: two breakers on the AVO loop (2026-09-06). Read notes/B-avo-… for the transcripts.
import inference.avo.agent as _avo
from pathlib import Path as _V1Path

_V1_DEAD_TURNS_NO_LEVEL = 4    # consecutive analyze() calls with step_executed False, game has cleared nothing
_V1_DEAD_TURNS_SCORING = 12    # same, once the game has cleared a level (three full AVO cycles; B60's lesson)
_V1_FORCED_CAP = 25            # forced actions per game, so a dead game cannot become a random walk
_V1_EXPLOIT_GAMEOVERS = 2      # GAME_OVERs inside exploit mode before exploit mode is switched off
_V1_SIMPLE = ("UP", "DOWN", "LEFT", "RIGHT", "SPACE")   # never MOUSE (needs coordinates), never RESET
_V1_STATS = {"games": 0, "dead_turns": 0, "forced": 0, "forced_executed": 0, "exploit_off": 0, "wrapper_errors": 0}
_orig_avo_analyze = _avo.AvoAgent.analyze


def _v1_game(state_path):
    try:
        return _V1Path(str(state_path)).stem.split("_")[0][:4]   # the #127 rule: the runtime dir is flat on Kaggle
    except Exception:
        return "????"


def _v1_state(agent):
    st = agent.__dict__.get("_v1_state")
    if st is None:
        st = agent.__dict__["_v1_state"] = {"dead": 0, "forced": 0, "gameovers": 0, "exploit_off": False}
        _V1_STATS["games"] += 1
    return st


def _v1_pick(valid_actions, n):
    names = [str(getattr(a, "name", a)).upper() for a in (valid_actions or [])]
    cand = [a for a in names if a in _V1_SIMPLE]
    return cand[n % len(cand)] if cand else None


def _v1_after_turn(agent, result, state_path, valid_actions, step_env):
    """The breakers. Pure function of the turn's result and the agent's own bookkeeping; drivable by teeth."""
    st = _v1_state(agent)
    gid = _v1_game(state_path)
    executed = bool(getattr(result, "step_executed", False))
    summ = agent._last_step_summary or {}
    # A: dead-game breaker
    if executed:
        st["dead"] = 0
    else:
        st["dead"] += 1
        _V1_STATS["dead_turns"] += 1
    best_level = int(getattr(getattr(agent, "supervisor", None), "best_level", 0) or 0)
    k = _V1_DEAD_TURNS_SCORING if best_level > 0 else _V1_DEAD_TURNS_NO_LEVEL
    if st["dead"] >= k and st["forced"] < _V1_FORCED_CAP and step_env is not None:
        act = _v1_pick(valid_actions, st["forced"])
        if act is not None:
            payload = step_env({"actions": [act]})
            ok = isinstance(payload, dict) and bool(payload.get("executed"))
            st["forced"] += 1
            st["dead"] = 0
            _V1_STATS["forced"] += 1
            _V1_STATS["forced_executed"] += int(ok)
            print(f"thui-avo-v1: game={gid} forced #{st['forced']} act={act} after {k} dead turns executed={ok} "
                  f"changed={payload.get('board_changed') if isinstance(payload, dict) else None} "
                  f"level_completed={payload.get('level_completed') if isinstance(payload, dict) else None}", flush=True)
    # B: exploit-replay breaker (only a turn that executed can carry a fresh game_over; the summary persists)
    if executed and bool(getattr(agent, "in_exploit_mode", False)) and bool(summ.get("game_over")):
        st["gameovers"] += 1
        if st["gameovers"] >= _V1_EXPLOIT_GAMEOVERS and not st["exploit_off"]:
            st["exploit_off"] = True
            agent.game_budget_s = None          # AvoAgent.in_exploit_mode reads False from here on
            _V1_STATS["exploit_off"] += 1
            try:
                agent.memory.record_failure(f"exploit-mode replay ended the game {st['gameovers']} times; that sequence is not a solution")
            except Exception:
                pass
            print(f"thui-avo-v1: game={gid} exploit mode OFF after {st['gameovers']} game-overs (phases resume)", flush=True)
    return st


def _v1_analyze(self, state_path, action_count, *args, valid_actions=None, step_env=None, **kwargs):
    result = _orig_avo_analyze(self, state_path, action_count, *args, valid_actions=valid_actions, step_env=step_env, **kwargs)
    if result is None or getattr(result, "retryable_failure", False):
        return result
    try:
        _v1_after_turn(self, result, state_path, valid_actions, step_env)
    except Exception as exc:   # the breakers must never break the harness path
        _V1_STATS["wrapper_errors"] += 1
        print(f"thui-avo-v1: wrapper error {type(exc).__name__}: {str(exc)[:160]}", flush=True)
    return result


_avo.AvoAgent.analyze = _v1_analyze
assert _avo.AvoAgent.analyze is _v1_analyze, "thui-avo-v1: analyze wrap did not land"

# teeth: drive the breakers on a fake agent with a stub step_env, before any game runs
class _V1Fake:
    def __init__(self):
        self._last_step_summary = None
        self.game_budget_s = 7920.0
        self.in_exploit_mode = False
        self.supervisor = type("S", (), {"best_level": 0})()
        self.memory = type("M", (), {"failures": [], "record_failure": lambda s, t: s.failures.append(t)})()
_calls = []
def _stub_step_env(arguments):
    _calls.append(arguments); return {"executed": True, "board_changed": True, "level_completed": False}
_R = type("R", (), {"step_executed": False, "retryable_failure": False})
_fa = _V1Fake(); _sp = "/kaggle/working/artifacts/m0r0-492f87ba_p0_tool_runtime_state.json"
for _i in range(_V1_DEAD_TURNS_NO_LEVEL - 1):
    _v1_after_turn(_fa, _R(), _sp, ["UP", "DOWN", "MOUSE"], _stub_step_env)
assert _calls == [], "thui-avo-v1 TEETH: forced before the threshold"
_v1_after_turn(_fa, _R(), _sp, ["UP", "DOWN", "MOUSE"], _stub_step_env)
assert _calls == [{"actions": ["UP"]}], f"thui-avo-v1 TEETH: forced action wrong: {_calls}"
assert _fa.__dict__["_v1_state"]["dead"] == 0 and _V1_STATS["forced"] == 1
_v1_after_turn(_fa, type("R", (), {"step_executed": True, "retryable_failure": False})(), _sp, ["UP"], _stub_step_env)
assert _fa.__dict__["_v1_state"]["dead"] == 0, "thui-avo-v1 TEETH: an executed turn must reset the dead counter"
_fa.supervisor.best_level = 1
for _i in range(_V1_DEAD_TURNS_NO_LEVEL):
    _v1_after_turn(_fa, _R(), _sp, ["UP"], _stub_step_env)
assert len(_calls) == 1, "thui-avo-v1 TEETH: a scoring game must use the 12-turn threshold, not 4"
_fb = _V1Fake(); _fb.in_exploit_mode = True; _fb._last_step_summary = {"game_over": True, "executed_count": 3}
_Rx = type("R", (), {"step_executed": True, "retryable_failure": False})
_v1_after_turn(_fb, _Rx(), _sp, ["UP"], _stub_step_env)
assert _fb.game_budget_s == 7920.0, "thui-avo-v1 TEETH: exploit switched off after ONE game-over"
_v1_after_turn(_fb, _Rx(), _sp, ["UP"], _stub_step_env)
assert _fb.game_budget_s is None and _fb.memory.failures and _V1_STATS["exploit_off"] == 1, "thui-avo-v1 TEETH: exploit not switched off after two"
_v1_after_turn(_fb, _R(), _sp, ["UP"], _stub_step_env)   # a non-executed turn cannot re-count the stale game_over
assert _fb.__dict__["_v1_state"]["gameovers"] == 2, "thui-avo-v1 TEETH: stale game_over re-counted on a no-action turn"
assert _v1_game(_sp) == "m0r0" and _v1_pick(["MOUSE"], 0) is None
for _k in _V1_STATS: _V1_STATS[_k] = 0
del _fa, _fb, _calls, _R, _Rx, _sp
print(f"thui-avo-v1: AvoAgent.analyze wrapped; breakers A (dead turns {_V1_DEAD_TURNS_NO_LEVEL}/{_V1_DEAD_TURNS_SCORING}, cap {_V1_FORCED_CAP}) "
      f"and B (exploit off after {_V1_EXPLOIT_GAMEOVERS} game-overs); teeth ok", flush=True)
# ======================================================================================


## 7. Run the benchmark

In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the Kaggle gateway and
play the **live competition Arcade**. Otherwise — an interactive "Save & Run" — play the
competition's **bundled environment files offline**, with no gateway required, so the notebook
runs end-to-end without a submission. Teardown commands run afterward even if the run raises.

In [ ]:
# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Print the run preamble and persist the launcher's git status for diagnostics.
print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
    bm.games = _offline_games(competition_env_files)

bm.n_passes = 1
bm.game_weights = None

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
soft_end = None
if not TRUE_SUBMISSION:
    budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    if budget > 0:
        soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=budget - min(600.0, budget / 2))

# Play the benchmark; teardown commands run even if the run raises.
try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
    if not TRUE_SUBMISSION:
        # An offline run isn't scored, but Kaggle still expects a submission.parquet output.
        import pandas as pd

        pd.DataFrame(
            [["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        ).to_parquet(WORKING_DIR / "submission.parquet", index=False)
finally:
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print(f"taaf.kaggle: teardown command: {command}", flush=True)
        subprocess.run(command, shell=True, check=False, cwd=WORKING_DIR, env=_command_env())

## 8. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")